# Configuração inicial

In [28]:
import pandas as pd
import numpy as np
import json
import re
import time
import os
from typing import Any, Dict, Optional
from datetime import datetime

# Configurações do Pandas
pd.set_option('display.max_columns', None)

# Regex para extrair domínio de e-mail
EMAIL_REGEX = re.compile(r"^[^@\s]+@([^@\s]+\.[^@\s]+)$")

# Caminho do arquivo da Camada Prata
FILE_PATH_SILVER = "../bases_de_dados/sintetica/df_prata_cleaned.csv"
FIXTURES_DIR = "services/inference/fixtures"

## Classe para enriquecimento

In [29]:
class EnrichmentService:
    def __init__(self, fixtures_dir: str = FIXTURES_DIR):
        """
        Inicializa o serviço, carregando os fixtures (mocks de API) do disco.
        """
        self.fixtures_dir = fixtures_dir
        
        # Carregar fixtures
        print(f"Carregando fixtures de: {self.fixtures_dir}")
        self._geo = self.load("geoip.json")
        self._bin = self.load("bin.json")
        self._dom = self.load("domains.json")
        
        # Exibir status do carregamento
        print(f"GeoIP carregado: {len(self._geo)} entradas.")
        print(f"BIN carregado: {len(self._bin)} entradas.")
        print(f"Domains carregado: {len(self._dom)} entradas.")

    def load(self, name: str) -> dict:
        """Função interna para carregar fixtures do disco."""
        try:
            path = os.path.join(self.fixtures_dir, name)
            if os.path.exists(path):
                with open(path, "r", encoding="utf-8") as f:
                    return json.load(f)
        except Exception as e:
            print(f"Aviso: Não foi possível carregar {name} ({e}). Usando dict vazio.")
        return {}

    def extract_email_domain(self, email: str) -> Optional[str]:
        if pd.isna(email): return None
        m = EMAIL_REGEX.match(str(email))
        return m.group(1).lower() if m else None

    # --- Funções de Lookup ---

    def get_geoip(self, ip: Optional[str]) -> Dict[str, Any]:
        if not ip or pd.isna(ip): return {"country_name": None, "city": None, "org": None}
        return self._geo.get(ip, {"country_name": None, "city": None, "org": None})

    def get_bin(self, card_bin_val: Optional[str]) -> Dict[str, Any]:
        if not card_bin_val or pd.isna(card_bin_val): 
            return {"scheme": None, "brand": None, "bank": {"name": None}, "country": {"name": None}}
            
        digits = re.sub(r"\D", "", str(card_bin_val))[:8]
        # Tenta 8 dígitos, se falhar, tenta 6 dígitos
        res = self._bin.get(digits) or self._bin.get(digits[:6])
        
        if res:
            return res
        return {"scheme": None, "brand": None, "bank": {"name": None}, "country": {"name": None}}

    def get_email_domain_info(self, email: Optional[str]) -> Dict[str, Any]:
        domain = self.extract_email_domain(email)
        if not domain:
            return {"domain": None, "has_mx": False, "disposable": False}
            
        entry = self._dom.get(domain, {}) 
        return {
            "domain": domain,
            "has_mx": bool(entry.get("dns", {}).get("has_mx", False)), 
            "disposable": bool(entry.get("disposable", False))
        }
    
    def get_doc_info(self, document: Optional[str]) -> Dict[str, Any]:
        """
        Função simplificada: Retorna o identificador do documento, se existir.
        """
        if not document:
            return {"user_document_id": None}
        
        # Simplesmente retorna o documento (ou hash) limpo
        cleaned_doc = re.sub(r"\D", "", str(document))
        
        return {
            "user_document_id": cleaned_doc,
        }

    def enrich_row(self, row):
        """Aplica todas as lógicas de enriquecimento a uma única linha do DataFrame."""
        # Se as colunas estiverem ausentes, o .get() retorna None, o que é tratado pelas funções de lookup
        ip = row.get("browser_ip", None) 
        card_bin = row.get("bin", None)
        email = row.get("email", None)
        card_doc = row.get("cardHolderDocument", None)

        geo_data = self.get_geoip(ip)
        bin_data = self.get_bin(card_bin)
        email_data = self.get_email_domain_info(email)
        doc_data = self.get_doc_info(card_doc)

        # Retorna pd.Series para expansão automática
        return pd.Series({
            # Estrutura mantida para uso futuro:
            #"enr_ip_country": geo_data.get("country_name"),
            #"enr_ip_city": geo_data.get("city"),
            #"enr_ip_org": geo_data.get("org"),
            #"enr_card_scheme": bin_data.get("scheme"),
            #"enr_card_brand": bin_data.get("brand"),
            #"enr_card_bank": bin_data.get("bank", {}).get("name"),
            #"enr_card_country": bin_data.get("country", {}).get("name"),
            
            # Estrutura preenchida:
            "enr_email_domain": email_data.get("domain"),
            "enr_email_is_disposable": email_data.get("disposable"),
            "enr_email_has_mx": email_data.get("has_mx"),
            "enr_user_doc_id": doc_data.get("user_document_id"),
        }, dtype='object')

## Histórico de Transações

In [30]:
def get_last_n_transactions(df_group: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    """
    Retorna as últimas N transações de um grupo (usuário).
    As transações devem ser pré-ordenadas.
    """
    return df_group.tail(n)

def calculate_history_features(df_history: pd.DataFrame) -> Dict[str, Any]:
    """
    Calcula features de histórico (mockado) a partir das N últimas transações.
    """
    # Se a coluna 'purchaseAmount' existir e for numérica
    if 'purchaseAmount' in df_history.columns and pd.api.types.is_numeric_dtype(df_history['purchaseAmount']):
        
        # 1. Contagem das últimas transações
        tx_count_last_10 = len(df_history)

        # 2. Média de valor das últimas 10 transações
        avg_amount_last_10 = round(df_history['purchaseAmount'].mean(), 2)

        # 3. Contagem de cartões únicos
        # Assumindo que 'acctNumber' ou 'cardExpiryDate' pode servir como proxy para o cartão
        distinct_cards_last_10 = df_history['cardExpiryDate'].nunique()
        
        return {
            "feat_doc_tx_count_last_10": tx_count_last_10,
            "feat_doc_avg_amount_last_10": avg_amount_last_10,
            "feat_doc_distinct_cards_last_10": distinct_cards_last_10,
        }
    else:
        # Retorna valores nulos se a coluna de valor não for numérica
        return {
            "feat_doc_tx_count_last_10": None,
            "feat_doc_avg_amount_last_10": None,
            "feat_doc_distinct_cards_last_10": None,
        }

## Enriquecimento dos dados

In [31]:
try:
    print(f"Lendo arquivo: {FILE_PATH_SILVER}")
    df_silver = pd.read_csv(FILE_PATH_SILVER, encoding='latin1', dtype={'purchaseDate': str}) 
except Exception as e:
    raise FileNotFoundError(f"Falha ao carregar {FILE_PATH_SILVER}. Erro: {e}")

enricher = EnrichmentService(fixtures_dir=FIXTURES_DIR)

print("\n1. Executando Enriquecimento Básico...")
df_enriched_columns = df_silver.apply(enricher.enrich_row, axis=1)
df_working = pd.concat([df_silver, df_enriched_columns], axis=1)

df_working['purchaseDateTime'] = pd.to_datetime(df_working['purchaseDate'], format='%Y%m%d%H%M%S', errors='coerce')
df_working.sort_values(by=['enr_user_doc_id', 'purchaseDateTime'], inplace=True)


cols_to_lag = [
    'purchaseAmount', 
    'merchantCountryCode', 
    'challengeWindowSize', 
    'purchaseInstalData'
]

for lag in [1, 2, 3]:
    for col in cols_to_lag:
        new_col_name = f'feat_lag_{col}_n-{lag}'
        df_working[new_col_name] = df_working.groupby('enr_user_doc_id')[col].shift(lag)


df_working['feat_time_since_last_tx_sec'] = df_working.groupby('enr_user_doc_id')['purchaseDateTime'].diff().dt.total_seconds()


def calculate_rolling_features(df_group):
    """Calcula features agregadas (Média, Soma, Std) das últimas 10 transações."""
    purchase_amount = pd.to_numeric(df_group['purchaseAmount'], errors='coerce')
    
    rolling_window = purchase_amount.rolling(window=10, min_periods=1)
    
    tx_sum = rolling_window.sum().shift(1).fillna(0)
    avg_amount = rolling_window.mean().shift(1)
    std_amount = rolling_window.std().shift(1)
    
    return pd.DataFrame({
        'feat_doc_tx_sum_last_10': tx_sum.round(2),
        'feat_doc_avg_amount_last_10': avg_amount.round(2),
        'feat_doc_std_amount_last_10': std_amount.round(2)
    })

df_rolling_features = df_working.groupby('enr_user_doc_id', dropna=False).apply(
    calculate_rolling_features
).reset_index(level=0, drop=True)


df_gold = pd.concat([df_working.drop(columns=['purchaseDateTime']), df_rolling_features], axis=1)


cols_numeric_lags = [c for c in df_gold.columns if 'feat_' in c]
df_gold[cols_numeric_lags] = df_gold[cols_numeric_lags].fillna(-1)

# O resto preenche com vazio
df_gold = df_gold.replace({np.nan: None}).fillna('')

print("✅ Engenharia de Features Completa.")

check_cols = ['enr_user_doc_id', 'purchaseAmount', 'feat_lag_purchaseAmount_n-1', 'feat_time_since_last_tx_sec', 'feat_doc_tx_sum_last_10']
print(df_gold[check_cols].head(10))

Lendo arquivo: ../bases_de_dados/sintetica/df_prata_cleaned.csv
Carregando fixtures de: services/inference/fixtures
GeoIP carregado: 0 entradas.
BIN carregado: 0 entradas.
Domains carregado: 0 entradas.

1. Executando Enriquecimento Básico...
✅ Engenharia de Features Completa.
     enr_user_doc_id  purchaseAmount  feat_lag_purchaseAmount_n-1  \
2513     10331849345           15509                         -1.0   
7502     10331849345           13894                      15509.0   
3542     10331849345            1500                      13894.0   
6228     10331849345            2112                       1500.0   
298      10331849345           31843                       2112.0   
1824     10331849345           21247                      31843.0   
5477     10331849345           31648                      21247.0   
1635     10331849345           10719                      31648.0   
3376     10331849345           23680                      10719.0   
245      10331849345           2

## Salvando os CSV/Parquet

In [32]:
base_name = "df_gold_enriched_" + datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = os.path.dirname(FILE_PATH_SILVER)

df_gold = df_gold.fillna('')


output_csv = os.path.join(output_dir, base_name + ".csv")
df_gold.to_csv(output_csv, index=False, encoding='utf-8')
print(f"✅ Camada Ouro salva em CSV: {output_csv}")


output_xlsx = os.path.join(output_dir, base_name + ".xlsx")
df_gold.to_excel(output_xlsx, index=False)
print(f"✅ Camada Ouro salva em XLSX: {output_xlsx}")

#output_parquet = os.path.join(output_dir, base_name + ".parquet")
#df_gold.to_parquet(output_parquet, index=False)
#print(f"✅ Camada Ouro salva em Parquet: {output_parquet}")


print("\nArquivos da Camada Ouro gerados com sucesso!")

✅ Camada Ouro salva em CSV: ../bases_de_dados/sintetica/df_gold_enriched_20251121_132933.csv
✅ Camada Ouro salva em XLSX: ../bases_de_dados/sintetica/df_gold_enriched_20251121_132933.xlsx

Arquivos da Camada Ouro gerados com sucesso!
